In [31]:
!uv pip install --upgrade pip
!uv pip install huggingface_hub accelerate scikit-learn
!uv pip uninstall torchvision pandas
!uv pip uninstall transformers tokenizers accelerate
!uv pip install "transformers==4.56.0" "protobuf==5.29.3"
!uv pip install torch datasets
!uv pip install tqdm wandb
!uv pip install bitsandbytes accelerate hf_transfer
!uv pip install --force-reinstall --no-cache-dir "numpy==2.0"

# Install PyTorch/XLA for TPU (Kaggle)
# !uv pip install torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html

Using Python 3.12.13 environment at: /usr
Resolved 1 package in 76ms
Checked 1 package in 0.18ms
Using Python 3.12.13 environment at: /usr
Checked 3 packages in 102ms
Using Python 3.12.13 environment at: /usr
Uninstalled 1 package in 104ms
 - pandas==3.0.2
Using Python 3.12.13 environment at: /usr
Uninstalled 3 packages in 218ms
 - accelerate==1.13.0
 - tokenizers==0.22.2
 - transformers==4.56.0
Using Python 3.12.13 environment at: /usr
Resolved 19 packages in 31ms
Installed 2 packages in 74ms
 + tokenizers==0.22.2
 + transformers==4.56.0
Using Python 3.12.13 environment at: /usr
Resolved 55 packages in 59ms
Installed 1 package in 41ms
 + pandas==3.0.2
Using Python 3.12.13 environment at: /usr
Checked 2 packages in 124ms
Using Python 3.12.13 environment at: /usr
Resolved 44 packages in 31ms
Installed 1 package in 6ms
 + accelerate==1.13.0
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 241ms
Prepared 1 package in 545ms
Uninstalled 1 package in 26ms
Installed 1 package i

In [32]:
import os
import re
import logging
import sys
import subprocess
from pathlib import Path

import torch
import torch.nn as nn
# import torch_xla
# import torch_xla.core.xla_model as xm

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

from IPython import get_ipython
import wandb
from typing import Dict, Tuple

def configure_environment_paths():
    try:
        if "google.colab" in str(get_ipython()):
            print("✅ Environment: Google Colab")
            base_data_path = "/content/"
            base_output_path = "/content/"
            environment_name = "colab"
        elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            print("✅ Environment: Kaggle")
            base_data_path = "/kaggle/input/"
            base_output_path = "/kaggle/working/"
            environment_name = "kaggle"
        else:
            print("⚠️ Environment: Local/Unknown")
            base_data_path = "./data/"
            base_output_path = "./output/"
            environment_name = "local"
    except NameError:
        print("⚠️ Non-interactive session. Using local paths.")
        base_data_path = "./data/"
        base_output_path = "./output/"
        environment_name = "local"
    os.makedirs(base_output_path, exist_ok=True)
    print(f"📂 Data Path: {base_data_path}")
    print(f"📦 Output Path: {base_output_path}")
    return base_data_path, base_output_path, environment_name

def load_secret(key_name: str, env_name: str) -> str | None:
    secret_value = None
    print(f"Attempting to load secret '{key_name}' from '{env_name}' environment...")
    try:
        if env_name == "colab":
            from google.colab import userdata
            secret_value = userdata.get(key_name)
        elif env_name == "kaggle":
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            secret_value = user_secrets.get_secret(key_name)
        else:
            secret_value = os.getenv(key_name)
        if not secret_value:
            print(f"⚠️ Secret '{key_name}' not found in the {env_name} environment.")
            return None
        print(f"✅ Successfully loaded secret '{key_name}'.")
        return secret_value
    except Exception as e:
        print(f"❌ An error occurred while loading secret '{key_name}': {e}")
        return None

def print_system_info():
    print("\n🔧 System Information")
    print(f"Python version: {sys.version.split()[0]}")
    try:
        import torch
        print(f"PyTorch version: {torch.__version__}")
        if torch.cuda.is_available():
            print(f"CUDA version: {torch.version.cuda}")
            print(f"GPU count: {torch.cuda.device_count()}")
            for i in range(torch.cuda.device_count()):
                print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        else:
            print("CUDA not available")
        # TPU detection
        try:
            import torch_xla
            print(f"PyTorch/XLA version: {torch_xla.__version__}")
            print(f"TPU cores: {xm.xrt_world_size()}")
        except:
            pass
    except ImportError:
        print("PyTorch not installed")

INPUT_PATH, OUTPUT_PATH, ENV_NAME = configure_environment_paths()
is_kaggle = "kaggle" in ENV_NAME
is_colab = not is_kaggle
print_system_info()

os.environ["WANDB_API_KEY"] = wandb_key = load_secret("WANDB_API_KEY", ENV_NAME)
os.environ["HF_TOKEN"] = HF_TOKEN = load_secret("HF_TOKEN", ENV_NAME)
GITHUB_TOKEN = load_secret("GITHUB_TOKEN", ENV_NAME)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import wandb
import huggingface_hub
wandb.login()
huggingface_hub.login(token=os.environ["HF_TOKEN"])


✅ Environment: Google Colab
📂 Data Path: /content/
📦 Output Path: /content/

🔧 System Information
Python version: 3.12.13
PyTorch version: 2.10.0+cu128
CUDA version: 12.8
GPU count: 1
  GPU 0: Tesla T4
Attempting to load secret 'WANDB_API_KEY' from 'colab' environment...
✅ Successfully loaded secret 'WANDB_API_KEY'.
Attempting to load secret 'HF_TOKEN' from 'colab' environment...
✅ Successfully loaded secret 'HF_TOKEN'.
Attempting to load secret 'GITHUB_TOKEN' from 'colab' environment...
✅ Successfully loaded secret 'GITHUB_TOKEN'.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [33]:
def download_from_hf(
    repo_id: str,
    local_dir: str = "ckpt",
    allow_patterns=None,
    force_download: bool = False,
    repo_type: str = "model",
):
    if allow_patterns is None:
        allow_patterns = ["*.safetensors"]
    print(f"📥 Downloading checkpoint from Hugging Face Hub to '{local_dir}'...\n")
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        repo_type=repo_type,
        allow_patterns=allow_patterns,
        force_download=force_download,
    )
    print("\n✅ Download complete!")
    print(f"\n📂 Files in {local_dir}/:")
    for file in os.listdir(local_dir):
        if file.endswith(".safetensors"):
            size = os.path.getsize(os.path.join(local_dir, file)) / (1024**2)
            print(f"  ✓ {file} ({size:.2f} MB)")

In [34]:
!rm -rf ckpt
download_from_hf(
    repo_id="dzungpham/graphcodebert-code-classification",
    local_dir="best_checkpoints",
    allow_patterns=["graphcodebert-robust/checkpoint-200*"],
)

📥 Downloading checkpoint from Hugging Face Hub to 'best_checkpoints'...



Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]


✅ Download complete!

📂 Files in best_checkpoints/:


In [35]:
# ============================================================================
# 1. DATA LAYER: Load and preprocess datasets
# ============================================================================

import random
import os
import re
import logging
import numpy as np
from typing import Dict, Tuple
from datasets import load_dataset, Dataset, concatenate_datasets

def setup_logger(output_dir: str, name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.handlers.clear()
    logger.setLevel(logging.INFO)
    logger.propagate = False
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
    logger.addHandler(console_handler)
    os.makedirs(output_dir, exist_ok=True)
    log_path = Path(output_dir) / f"{name}.log"
    file_handler = logging.FileHandler(log_path, mode="w")
    file_handler.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
    logger.addHandler(file_handler)
    return logger
logger = setup_logger(".", "notebook")

def set_seed(seed: int):
    """Set all random seeds for reproducibility (including TPU if available)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # For TPU, seed is handled by torch.manual_seed
    os.environ['PYTHONHASHSEED'] = str(seed)

def preprocess_code(code_str: str, seed: int = None) -> str:
    """Normalize code string with semantic-preserving perturbations for training.
       Uses the global random state; seed must be set before calling."""
    code_str = code_str.lstrip("\ufeff\u200b\u200c\u200d")
    code_str = re.sub(r"\r\n|\r", "\n", code_str)

    # 20% chance to replace 4 spaces with tabs
    if random.random() < 0.2:
        code_str = code_str.replace("    ", "\t")
    # 20% chance to replace tabs with 2 spaces
    elif random.random() < 0.2:
        code_str = code_str.replace("    ", "  ")

    code_str = "\n".join(line.rstrip() for line in code_str.split("\n"))
    code_str = re.sub(r"\n{3,}", "\n\n", code_str)
    code_str = re.sub(r"[ \t]+", " ", code_str)
    return code_str.strip()

def tokenize_function(examples: Dict, tokenizer, max_length: int = 512) -> Dict:
    """Tokenize code examples."""
    codes = [preprocess_code(c) for c in examples["code"]]
    return tokenizer(codes, truncation=True, max_length=max_length, padding=False)

def upsample_dataset(dataset: Dataset) -> Dataset:
    """Upsample the minority class to match the majority class size."""
    labels = np.array(dataset["label"])
    class_counts = np.bincount(labels)
    majority_class = np.argmax(class_counts)
    minority_class = np.argmin(class_counts)

    logger.info(f"Initial distribution: Class 0: {class_counts[0]}, Class 1: {class_counts[1]}")

    diff = class_counts[majority_class] - class_counts[minority_class]

    if diff > 0:
        minority_indices = np.where(labels == minority_class)[0]
        upsample_indices = np.random.choice(minority_indices, size=diff, replace=True)
        upsampled_data = dataset.select(upsample_indices)
        dataset = concatenate_datasets([dataset, upsampled_data])

        new_counts = np.bincount(np.array(dataset["label"]))
        logger.info(f"Balanced distribution: Class 0: {new_counts[0]}, Class 1: {new_counts[1]}")

    return dataset.shuffle(seed=42)

def load_datasets(tokenizer, max_length: int = 512, aug=None) -> Tuple[Dataset, Dataset]:
    """Load and tokenize train and validation datasets with upsampling."""
    logger.info("Loading datasets from Hugging Face Hub...")
    train_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="train")
    val_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="validation")

    logger.info(f"Train samples (before upsampling): {len(train_dataset)}")
    # Uncomment if upsampling desired:
    # train_dataset = upsample_dataset(train_dataset)

    logger.info(f"Final samples: Train: {len(train_dataset)}, Val: {len(val_dataset)}")

    if aug is not None:
        logger.info("Applying data augmentation before tokenization...")
        def augment_batch(examples):
            examples["code"] = [aug(c) for c in examples["code"]]
            return examples
        train_dataset = train_dataset.map(
            augment_batch,
            batched=True,
            batch_size=512,
            desc="Augmenting train",
            num_proc=os.cpu_count(),
        )

    # Tokenize
    logger.info("Tokenizing datasets...")
    train_dataset = train_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in train_dataset.column_names],
        desc="Tokenizing train",
        num_proc=os.cpu_count(),
    )

    val_dataset = val_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in val_dataset.column_names],
        desc="Tokenizing val",
        num_proc=os.cpu_count(),
    )

    # Rename label column
    train_dataset = train_dataset.rename_column("label", "labels")
    val_dataset = val_dataset.rename_column("label", "labels")

    return train_dataset, val_dataset

def compute_metrics_fn(eval_pred):
    """Compute precision, recall, F1, and accuracy for evaluation."""
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, predictions)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "macro_f1": f1
    }


In [36]:
# ============================================================================
# 2. MODEL LAYER: Load pretrained CodeBERT and tokenizer
# ============================================================================

def load_model_and_tokenizer(model_name: str, num_labels: int = 2):
    """Load pretrained model and tokenizer from Hugging Face."""
    logger.info(f"Loading tokenizer from: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    logger.info(f"Loading model from: {model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        problem_type="single_label_classification",
    )
    logger.info(f"Model loaded successfully")
    logger.info(f"Model config - num_labels: {model.config.num_labels}, hidden_size: {model.config.hidden_size}")
    return model, tokenizer

def freeze_base_model(model):
    """
    Freeze all base model parameters, keeping only classifier head trainable.
    """
    for name, param in model.named_parameters():
        if "classifier" not in name and "cls" not in name.lower():
            param.requires_grad = False
    for name, param in model.named_parameters():
        if "classifier" in name or "cls" in name.lower():
            param.requires_grad = True
    logger.info("Base model frozen - only classifier head is trainable")

In [37]:
def log_training_config(cfg, logger):
    logger.info("===== Training Configuration =====")
    for field_name, field_value in cfg.__dataclass_fields__.items():
        value = getattr(cfg, field_name)
        if field_name == "device":
            continue
        logger.info(f"{field_name:20} : {value}")
    logger.info("=================================")

def log_model_architecture(model, tokenizer, logger):
    logger.info("===== Model Architecture =====")
    logger.info("\n" + model.__repr__())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_params = trainable_params + non_trainable_params
    logger.info("===== Parameter Summary =====")
    logger.info(f"Total Parameters:         {total_params:,}")
    logger.info(f"Trainable Parameters:     {trainable_params:,}")
    logger.info(f"Non-trainable Parameters: {non_trainable_params:,}")
    logger.info("===== Tokenizer Summary =====")
    logger.info(f"Vocab size: {len(tokenizer)} | Special tokens: {tokenizer.all_special_tokens}")
    logger.info("===== End of Architecture Log =====")

In [38]:
# =============================================================================
# 3. TRAINING ENGINE: HF Trainer with MixCode + FFT low-pass + frequency consistency
# =============================================================================

import os
import logging
import random
import numpy as np
from dataclasses import dataclass, field
from typing import Callable, Optional, Tuple, List, Dict
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import (
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    PreTrainedTokenizer,
)

# ------------------------------
# FFT low-pass filter function
# ------------------------------
def low_pass_filter_embeddings(embeddings, keep_ratio=0.5):
    """
    Apply low‑pass filter to embeddings in the frequency domain.
    Args:
        embeddings: torch.Tensor of shape (batch, seq_len, hidden_dim)
        keep_ratio: fraction of low frequencies to keep (0.0 to 1.0)
    Returns:
        filtered_embeddings: same shape as input
    """
    batch, seq_len, hidden = embeddings.shape
    # FFT along sequence dimension (dim=1)
    fft = torch.fft.rfft(embeddings, dim=1, norm='ortho')
    keep_len = max(1, int(seq_len * keep_ratio))
    fft[:, keep_len:] = 0.0
    filtered = torch.fft.irfft(fft, n=seq_len, dim=1, norm='ortho')
    return filtered

# ------------------------------
# Enhanced CodeAugmentation with mixup parameters
# ------------------------------
class CodeAugmentation:
    def __init__(self, rename_prob=0.3, format_prob=0.3, mixup_alpha=1.0):
        self.rename_prob = rename_prob
        self.format_prob = format_prob
        self.mixup_alpha = mixup_alpha   # Beta distribution parameter for MixCode

    def __call__(self, text: str) -> str:
        text = str(text)
        if random.random() < self.rename_prob:
            import re
            tokens = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', text)
            keywords = {'def', 'return', 'if', 'else', 'for', 'while', 'import',
                        'from', 'as', 'with', 'try', 'except', 'class', 'lambda',
                        'and', 'or', 'not', 'True', 'False', 'None'}
            identifiers = [t for t in set(tokens) if t not in keywords and len(t) > 1]
            if identifiers:
                rename_map = {id_: f"var_{random.randint(1000,9999)}" for id_ in identifiers[:5]}
                for old, new in rename_map.items():
                    text = text.replace(old, new)
        if random.random() < self.format_prob:
            lines = text.split('\n')
            new_lines = []
            for line in lines:
                new_lines.append(line)
                if random.random() < 0.1:
                    new_lines.append('')
            text = '\n'.join(new_lines)
            if random.random() < 0.2:
                text = '\n'.join(' ' + line if random.random() < 0.1 else line
                                 for line in text.split('\n'))
        return text

# ------------------------------
# Loss function factories (unchanged)
# ------------------------------
def get_label_smoothed_cross_entropy(smoothing: float) -> Callable:
    def loss_fn(outputs, labels, **_):
        logits = outputs.logits
        n_classes = logits.size(-1)
        with torch.no_grad():
            smooth_targets = torch.zeros_like(logits).fill_(smoothing / (n_classes - 1))
            smooth_targets.scatter_(1, labels.unsqueeze(1), 1.0 - smoothing)
        return -(smooth_targets * F.log_softmax(logits, dim=-1)).sum(dim=-1).mean()
    return loss_fn

def get_focal_loss(alpha: float, gamma: float, smoothing: float = 0.0) -> Callable:
    def focal_loss(outputs, labels, **_):
        logits = outputs.logits
        if smoothing > 0:
            n_classes = logits.size(-1)
            smooth_targets = torch.zeros_like(logits).fill_(smoothing / (n_classes - 1))
            smooth_targets.scatter_(1, labels.unsqueeze(1), 1.0 - smoothing)
            ce = -(smooth_targets * F.log_softmax(logits, dim=-1)).sum(dim=-1)
        else:
            ce = F.cross_entropy(logits, labels, reduction="none")
        pt = torch.exp(-ce)
        loss = alpha * (1 - pt) ** gamma * ce
        return loss.mean()
    return focal_loss

def get_infonce_loss(temperature: float, weight: float) -> Callable:
    def infonce_loss(outputs, labels, **_):
        reps = outputs.hidden_states[-1]
        reps = F.normalize(reps, dim=-1)
        sim_matrix = torch.mm(reps, reps.t()) / temperature
        target = torch.arange(reps.size(0), device=reps.device)
        loss = F.cross_entropy(sim_matrix, target)
        return weight * loss
    return infonce_loss

# ------------------------------
# RobustTrainer with MixCode and frequency consistency
# ------------------------------
class RobustTrainer(Trainer):
    def __init__(
        self,
        *args,
        loss_type: str = "ce",
        r_drop_alpha: float = 4.0,
        compute_loss_fn: Optional[Callable] = None,
        label_smoothing: float = 0.0,
        adversarial_epsilon: float = 0.0,
        use_swa: bool = False,
        swa_start_epoch: int = 2,
        swa_lr: float = 1e-5,
        mixup_alpha: float = 1.0,
        low_pass_keep_ratio: float = 0.5,
        freq_consistency_weight: float = 0.1,
        **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.loss_type = loss_type
        self.r_drop_alpha = r_drop_alpha
        self.compute_loss_fn = compute_loss_fn
        self.label_smoothing = label_smoothing
        self.adversarial_epsilon = adversarial_epsilon
        self.use_swa = use_swa
        self.swa_start_epoch = swa_start_epoch
        self.swa_lr = swa_lr
        self.swa_model = None
        self.swa_scheduler = None
        self._swa_started = False
        # New parameters
        self.mixup_alpha = mixup_alpha
        self.low_pass_keep_ratio = low_pass_keep_ratio
        self.freq_consistency_weight = freq_consistency_weight

    def compute_loss(self, model, inputs, num_items_in_batch=64, return_outputs=False):
        labels = inputs.pop("labels", None)

        # ---- R-Drop (unchanged) ----
        if self.loss_type == "r-drop" and model.training and labels is not None:
            out1 = model(**inputs)
            out2 = model(**inputs)
            ce1 = F.cross_entropy(out1.logits, labels)
            ce2 = F.cross_entropy(out2.logits, labels)
            ce = (ce1 + ce2) / 2
            p = F.log_softmax(out1.logits, dim=-1)
            q = F.log_softmax(out2.logits, dim=-1)
            kl = (F.kl_div(p, F.softmax(out2.logits, dim=-1), reduction="batchmean") +
                  F.kl_div(q, F.softmax(out1.logits, dim=-1), reduction="batchmean")) / 2
            loss = ce + self.r_drop_alpha * kl
            return (loss, out1) if return_outputs else loss

        # ---- MixCode (if enabled) ----
        if self.mixup_alpha > 0 and model.training and labels is not None:
            batch_size = inputs['input_ids'].size(0)
            device = inputs['input_ids'].device
            shuffle_idx = torch.randperm(batch_size, device=device)
            lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)

            # Get embedding layer (GraphCodeBERT uses roberta)
            emb_layer = model.roberta.embeddings.word_embeddings
            orig_embeds = emb_layer(inputs['input_ids'])
            shuffled_embeds = emb_layer(inputs['input_ids'][shuffle_idx])

            mixed_embeds = lam * orig_embeds + (1 - lam) * shuffled_embeds
            mixed_attn = inputs['attention_mask']

            outputs = model(inputs_embeds=mixed_embeds, attention_mask=mixed_attn)

            # Soft labels
            n_classes = model.config.num_labels
            one_hot = F.one_hot(labels, n_classes).float()
            shuffled_labels = labels[shuffle_idx]
            one_hot_shuffled = F.one_hot(shuffled_labels, n_classes).float()
            mixed_labels = lam * one_hot + (1 - lam) * one_hot_shuffled

            loss = -(mixed_labels * F.log_softmax(outputs.logits, dim=-1)).sum(dim=-1).mean()
        else:
            outputs = model(**inputs)
            if labels is None:
                loss = outputs.loss if hasattr(outputs, "loss") else outputs[0]
            else:
                if self.compute_loss_fn:
                    loss = self.compute_loss_fn(outputs, labels)
                else:
                    if self.label_smoothing > 0 and self.loss_type == "ce":
                        loss_fn = get_label_smoothed_cross_entropy(self.label_smoothing)
                        loss = loss_fn(outputs, labels)
                    else:
                        loss = F.cross_entropy(outputs.logits, labels)

        # ---- Frequency-domain consistency loss (FFT low-pass) ----
        if self.freq_consistency_weight > 0 and model.training and labels is not None:
            emb_layer = model.roberta.embeddings.word_embeddings
            orig_embeds = emb_layer(inputs['input_ids'])
            filtered_embeds = low_pass_filter_embeddings(orig_embeds, self.low_pass_keep_ratio)
            filtered_outputs = model(inputs_embeds=filtered_embeds, attention_mask=inputs['attention_mask'])
            kl = F.kl_div(
                F.log_softmax(outputs.logits, dim=-1),
                F.softmax(filtered_outputs.logits, dim=-1),
                reduction='batchmean'
            )
            loss = loss + self.freq_consistency_weight * kl

        return (loss, outputs) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch):
        """Override to add FGM adversarial training and SWA updates."""
        # Standard forward + backward (first pass)
        loss = super().training_step(model, inputs)

        # ---- FGM Adversarial Training ----
        if self.adversarial_epsilon > 0 and model.training:
            # Clone inputs to avoid modifying original
            adv_inputs = {k: v.clone() if torch.is_tensor(v) else v for k, v in inputs.items()}
            labels = adv_inputs.get("labels", None)
            if labels is not None:
                # 1. Clean backward already done, so gradients exist.
                # 2. Get embeddings (handle different model architectures)
                if hasattr(model, 'roberta'):
                    embeddings = model.roberta.embeddings.word_embeddings
                elif hasattr(model, 'bert'):
                    embeddings = model.bert.embeddings.word_embeddings
                else:
                    embeddings = None
                if embeddings is not None:
                    grad = embeddings.weight.grad
                    if grad is not None:
                        perturbation = self.adversarial_epsilon * grad.sign()
                        original_emb = embeddings.weight.data.clone()
                        embeddings.weight.data.add_(perturbation)

                        # Forward with perturbed embeddings
                        adv_outputs = model(**{k:v for k,v in adv_inputs.items() if k != 'labels'})
                        if self.compute_loss_fn:
                            adv_loss = self.compute_loss_fn(adv_outputs, labels)
                        else:
                            if self.label_smoothing > 0 and self.loss_type == "ce":
                                loss_fn = get_label_smoothed_cross_entropy(self.label_smoothing)
                                adv_loss = loss_fn(adv_outputs, labels)
                            else:
                                adv_loss = F.cross_entropy(adv_outputs.logits, labels)

                        # Backward for adversarial loss (accumulate)
                        self.accelerator.backward(adv_loss)

                        # Restore embeddings
                        embeddings.weight.data = original_emb

        # ---- SWA Initialization and Update ----
        if self.use_swa and not self._swa_started and self.state.epoch >= self.swa_start_epoch:
            self._swa_started = True
            self.swa_model = AveragedModel(model)
            self.swa_scheduler = SWALR(self.optimizer, swa_lr=self.swa_lr)
            self.log({"SWA": "started"})

        if self._swa_started and self.swa_model is not None:
            self.swa_model.update_parameters(model)

        return loss

    def optimizer_step(self, model, optimizer, optimizer_idx=None):
        super().optimizer_step(model, optimizer, optimizer_idx)
        if self._swa_started and self.swa_scheduler is not None:
            self.swa_scheduler.step()

    def save_model(self, output_dir: Optional[str] = None, _internal_call: bool = False):
        if self._swa_started and self.swa_model is not None:
            if self.train_dataset is not None:
                train_loader = self.get_train_dataloader()
                update_bn(train_loader, self.swa_model)
            model_to_save = self.swa_model.module if hasattr(self.swa_model, 'module') else self.swa_model
            model_to_save.save_pretrained(output_dir or self.args.output_dir)
        else:
            super().save_model(output_dir, _internal_call)

# ------------------------------
# Updated TrainConfig with new parameters
# ------------------------------
@dataclass
class TrainConfig:
    model_name: str = "microsoft/graphcodebert-base"
    output_dir: str = "./graphcodebert"
    num_epochs: int = 3
    batch_size: int = 16
    learning_rate: float = 2e-5
    max_length: int = 512
    num_labels: int = 2
    use_wandb: bool = False
    freeze_base: bool = False
    loss_type: str = "r-drop"
    focal_alpha: float = 1.0
    focal_gamma: float = 2.0
    r_drop_alpha: float = 4.0
    infonce_temperature: float = 0.07
    infonce_weight: float = 0.5
    seed: int = 42
    resume_from_checkpoint: str = None

    label_smoothing: float = 0.1
    adversarial_epsilon: float = 0.5
    use_swa: bool = True
    swa_start_epoch: int = 2
    swa_lr: float = 1e-5
    data_augmentation: bool = True
    aug_rename_prob: float = 0.3
    aug_format_prob: float = 0.3

    # New parameters for MixCode and frequency domain
    mixup_alpha: float = 1.0
    low_pass_keep_ratio: float = 0.5
    freq_consistency_weight: float = 0.1

    device: torch.device = field(init=False, default_factory=lambda: torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# ------------------------------
# Main training pipeline (updated)
# ------------------------------
def train_pipeline(cfg: TrainConfig):
    os.makedirs(cfg.output_dir, exist_ok=True)
    logger = setup_logger(cfg.output_dir, "training")

    set_seed(cfg.seed)

    log_training_config(cfg, logger)

    # Load model & tokenizer
    model, tokenizer = load_model_and_tokenizer(cfg.model_name, cfg.num_labels)
    # Handle TPU device if available
    if "xla" in str(cfg.device) or (hasattr(torch, 'xla') and xm.xrt_world_size() > 1):
        device = xm.xla_device()
        logger.info("Using TPU device")
    else:
        device = cfg.device
    model.to(device)
    logger.info(f"Model placed on {device}")

    if cfg.freeze_base:
        freeze_base_model(model)

    log_model_architecture(model, tokenizer, logger)

    if cfg.loss_type == "infonce":
        model.config.output_hidden_states = True
        logger.info("Enabled hidden states for InfoNCE loss.")

    # Data augmentation (now includes mixup_alpha, but that's handled in trainer)
    aug = None
    if cfg.data_augmentation:
        aug = CodeAugmentation(rename_prob=cfg.aug_rename_prob,
                               format_prob=cfg.aug_format_prob,
                               mixup_alpha=cfg.mixup_alpha)
        logger.info(f"Data augmentation enabled (rename={cfg.aug_rename_prob}, format={cfg.aug_format_prob})")

    train_dataset, val_dataset = load_datasets(tokenizer, cfg.max_length, aug=aug)

    if cfg.use_wandb:
        try:
            import wandb
            os.environ["WANDB_MODE"] = "online"
        except Exception as e:
            logger.warning(f"W&B import failed ({e}); proceeding without it.")
            cfg.use_wandb = False

    steps_per_epoch = max(1, len(train_dataset) // cfg.batch_size)
    total_steps = cfg.num_epochs * steps_per_epoch
    warmup_steps = max(1, int(total_steps * 0.1))

    training_args = TrainingArguments(
        output_dir=cfg.output_dir,
        num_train_epochs=cfg.num_epochs,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size * 2,
        learning_rate=cfg.learning_rate,
        warmup_steps=warmup_steps,
        weight_decay=0.1,
        logging_strategy="steps",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=1000,
        save_strategy="steps",
        save_steps=100,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=4,
        report_to=["wandb"] if cfg.use_wandb else [],
        run_name="graphcodebert-robust" if cfg.use_wandb else None,
        dataloader_pin_memory=torch.cuda.is_available(),
        seed=cfg.seed,
        fp16=False if "xla" in str(device) else True,  # TPU uses bf16, not fp16
        bf16=True if "xla" in str(device) else False,
    )

    data_collator = DataCollatorWithPadding(tokenizer)

    compute_loss_fn = None
    if cfg.loss_type == "focal":
        compute_loss_fn = get_focal_loss(cfg.focal_alpha, cfg.focal_gamma, smoothing=cfg.label_smoothing if cfg.loss_type=="ce" else 0.0)
    elif cfg.loss_type == "infonce":
        compute_loss_fn = get_infonce_loss(cfg.infonce_temperature, cfg.infonce_weight)

    trainer = RobustTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics_fn,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        loss_type=cfg.loss_type,
        r_drop_alpha=cfg.r_drop_alpha,
        compute_loss_fn=compute_loss_fn,
        label_smoothing=cfg.label_smoothing,
        adversarial_epsilon=cfg.adversarial_epsilon,
        use_swa=cfg.use_swa,
        swa_start_epoch=cfg.swa_start_epoch,
        swa_lr=cfg.swa_lr,
        mixup_alpha=cfg.mixup_alpha,
        low_pass_keep_ratio=cfg.low_pass_keep_ratio,
        freq_consistency_weight=cfg.freq_consistency_weight,
    )

    logger.info("=== Starting training with MixCode + FFT low-pass consistency ===")
    trainer.train(resume_from_checkpoint=cfg.resume_from_checkpoint)
    logger.info("Training complete!")

    final_model = trainer.swa_model if (trainer._swa_started and trainer.swa_model) else model
    final_model.save_pretrained(os.path.join(cfg.output_dir, "final_model"))
    tokenizer.save_pretrained(os.path.join(cfg.output_dir, "final_model"))
    logger.info(f"Final model saved to {os.path.join(cfg.output_dir, 'final_model')}")

    return trainer, final_model, tokenizer


In [ ]:
if __name__ == "__main__":
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    cfg = TrainConfig(
        model_name="microsoft/graphcodebert-base",
        output_dir="output_checkpoints/graphcodebert-mixcode-fft",
        num_epochs=0.001,
        batch_size=64,
        learning_rate=5e-5,
        max_length=512,
        use_wandb=True,
        freeze_base=True,
        loss_type="r-drop",
        r_drop_alpha=6.0,
        label_smoothing=0.2,
        adversarial_epsilon=0.5,
        use_swa=True,
        swa_start_epoch=2,
        swa_lr=1e-5,
        data_augmentation=True,
        aug_rename_prob=0.6,
        aug_format_prob=0.6,
        # New parameters (best checkpoints 200 curently dont use these ones yet)
        mixup_alpha=1.0,
        low_pass_keep_ratio=0.5,
        freq_consistency_weight=0.1,

        resume_from_checkpoint=None,
    )
    trainer, model, tokenizer = train_pipeline(cfg)


2026-04-17 15:29:31,415 - INFO - ===== Training Configuration =====
2026-04-17 15:29:31,417 - INFO - model_name           : microsoft/graphcodebert-base
2026-04-17 15:29:31,418 - INFO - output_dir           : output_checkpoints/graphcodebert-mixcode-fft
2026-04-17 15:29:31,419 - INFO - num_epochs           : 0.001
2026-04-17 15:29:31,420 - INFO - batch_size           : 64
2026-04-17 15:29:31,422 - INFO - learning_rate        : 5e-05
2026-04-17 15:29:31,422 - INFO - max_length           : 512
2026-04-17 15:29:31,424 - INFO - num_labels           : 2
2026-04-17 15:29:31,424 - INFO - use_wandb            : True
2026-04-17 15:29:31,425 - INFO - freeze_base          : True
2026-04-17 15:29:31,426 - INFO - loss_type            : r-drop
2026-04-17 15:29:31,426 - INFO - focal_alpha          : 1.0
2026-04-17 15:29:31,427 - INFO - focal_gamma          : 2.0
2026-04-17 15:29:31,428 - INFO - r_drop_alpha         : 6.0
2026-04-17 15:29:31,429 - INFO - infonce_temperature  : 0.07
2026-04-17 15:29:31

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


2026-04-17 15:29:32,553 - INFO - Model loaded successfully
2026-04-17 15:29:32,554 - INFO - Model config - num_labels: 2, hidden_size: 768
2026-04-17 15:29:32,689 - INFO - Model placed on cuda
2026-04-17 15:29:32,691 - INFO - Base model frozen - only classifier head is trainable
2026-04-17 15:29:32,692 - INFO - ===== Model Architecture =====
2026-04-17 15:29:32,694 - INFO - 
RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=7

README.md:   0%|          | 0.00/801 [00:00<?, ?B/s]

task_a/task_a_training_set_1.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

2026-04-17 15:29:44,317 - INFO - Train samples (before upsampling): 500000
2026-04-17 15:29:44,319 - INFO - Final samples: Train: 500000, Val: 100000
2026-04-17 15:29:44,320 - INFO - Applying data augmentation before tokenization...


Augmenting train (num_proc=2):   0%|          | 0/500000 [00:00<?, ? examples/s]

2026-04-17 15:30:08,148 - INFO - Tokenizing datasets...


Tokenizing train (num_proc=2):   0%|          | 0/500000 [00:00<?, ? examples/s]

In [ ]:
import os
import logging
from pathlib import Path
from typing import Tuple, Dict, Any
import numpy as np
import torch
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

def log_model_architecture_inference(model, tokenizer, logger):
    logger.info("===== Model Architecture =====")
    logger.info("\n" + model.__repr__())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_params = trainable_params + non_trainable_params
    logger.info("===== Parameter Summary =====")
    logger.info(f"Total Parameters:         {total_params:,}")
    logger.info(f"Trainable Parameters:     {trainable_params:,}")
    logger.info(f"Non-trainable Parameters: {non_trainable_params:,}")
    logger.info("===== Tokenizer Summary =====")
    logger.info(f"Vocab size: {len(tokenizer)} | Special tokens: {tokenizer.all_special_tokens}")
    logger.info("===== End of Architecture Log =====")

def run_standalone_inference(
    checkpoint_path: str,
    output_dir: str = "./",
    output_csv: str = "submission.csv",
    batch_size: int = 32,
    max_length: int = 512,
    dataset_name: str = "DaniilOr/SemEval-2026-Task13",
    dataset_config: str = "A",
    split: str = "test",
):
    logger = setup_logger(output_dir, "inference")
    logger.info(f"Loading model and tokenizer from: {checkpoint_path}")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
    log_model_architecture_inference(model, tokenizer, logger)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    logger.info(f"Loading dataset: {dataset_name} ({dataset_config})")
    try:
        test_ds = load_dataset(dataset_name, dataset_config, split=split)
    except Exception as e:
        logger.warning(f"Default loading failed: {e}")
        logger.info(f"Attempting to load split '{split}' using data_files...")
        test_ds = load_dataset(dataset_name, data_files={split: f"*{split}*"}, split=split)

    def tokenize_fn(examples):
        return tokenizer(
            examples["code"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    logger.info("Tokenizing dataset...")
    tokenized_ds = test_ds.map(
        tokenize_fn,
        batched=True,
        remove_columns=[c for c in test_ds.column_names if c not in ["input_ids", "attention_mask", "id", "label"]],
        desc="Tokenizing"
    )
    tokenized_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

    test_loader = torch.utils.data.DataLoader(
        tokenized_ds,
        batch_size=batch_size,
        shuffle=False,
    )

    logger.info(f"Running inference on {len(test_ds)} examples...")
    all_logits = []
    with torch.inference_mode():
        for batch in tqdm(test_loader, desc="Predicting"):
            inputs = {k: v.to(device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}
            outputs = model(**inputs)
            all_logits.append(outputs.logits.cpu())
    logits = torch.cat(all_logits, dim=0).numpy()
    pred_labels = logits.argmax(axis=-1)

    if "label" in test_ds.column_names:
        logger.info("Calculating classification metrics...")
        true_labels = np.array(test_ds["label"])
        acc = accuracy_score(true_labels, pred_labels)
        precision, recall, f1, _ = precision_recall_fscore_support(
            true_labels, pred_labels, average='weighted'
        )
        logger.info("-" * 30)
        logger.info(f"METRICS FOR SPLIT: {split}")
        logger.info(f"Accuracy:  {acc:.4f}")
        logger.info(f"Precision: {precision:.4f}")
        logger.info(f"Recall:    {recall:.4f}")
        logger.info(f"F1-Score:  {f1:.4f}")
        logger.info("-" * 30)
        cm = confusion_matrix(true_labels, pred_labels)
        logger.info(f"Confusion Matrix:\n{cm}")
    else:
        logger.warning("No 'label' column found. Skipping metric calculation.")

    csv_path = Path(output_dir) / output_csv
    if "id" in test_ds.column_names:
        ids = test_ds["id"]
    elif "ID" in test_ds.column_names:
        ids = test_ds["ID"]
    else:
        ids = list(range(len(pred_labels)))
    with open(csv_path, "w", encoding="utf-8") as f:
        f.write("id,label\n")
        for idx, label in zip(ids, pred_labels):
            f.write(f"{idx},{label}\n")
    logger.info(f"✅ Predictions saved to {csv_path}")

In [ ]:
from pathlib import Path
import os
from datetime import datetime

def list_directory_sorted(directory: str, reverse_time: bool = True, show_hidden: bool = False):
    """
    List directory contents sorted by modification time (oldest first by default).

    Parameters:
        directory (str): Path to the directory to list.
        reverse_time (bool): If True, sort oldest first (like ls -lrt). If False, newest first.
        show_hidden (bool): Whether to include hidden files (starting with '.').

    Returns:
        None (prints formatted table)
    """
    path = Path(directory).expanduser().resolve()

    if not path.exists():
        print(f"❌ Directory does not exist: {path}")
        return

    if not path.is_dir():
        print(f"❌ Path is not a directory: {path}")
        return

    # Gather all items (files/directories)
    items = []
    for item in path.iterdir():
        if not show_hidden and item.name.startswith('.'):
            continue
        stat = item.stat()
        items.append({
            'name': item.name + ('/' if item.is_dir() else ''),
            'size': stat.st_size,
            'mtime': stat.st_mtime,
            'is_dir': item.is_dir()
        })

    # Sort by modification time (oldest first if reverse_time=True)
    items.sort(key=lambda x: x['mtime'], reverse=not reverse_time)

    # Print header
    print(f"\n📁 Contents of {path}\n")
    print(f"{'Permissions':<12} {'Size (bytes)':>12} {'Modified Time':<20} {'Name'}")
    print("-" * 70)

    for item in items:
        # Get permissions (simplified)
        perms = "drwxr-xr-x" if item['is_dir'] else "-rw-r--r--"
        # Format modification time
        mtime_str = datetime.fromtimestamp(item['mtime']).strftime("%Y-%m-%d %H:%M:%S")
        # Print row
        print(f"{perms:<12} {item['size']:>12,} {mtime_str:<20} {item['name']}")

    print(f"\n✅ Total items: {len(items)}")

# Usage example for your specific directory
list_directory_sorted("output_checkpoints/graphcodebert-robust")
list_directory_sorted("checkpoints/graphcodebert-robust")

In [ ]:
if __name__ == "__main__":
    CHECKPOINT = "output_checkpoints/graphcodebert-robust/checkpoint-400"
    OUTPUT_DIR = "test/inference/graphcodebert-robust"
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    if os.path.exists(CHECKPOINT):
        run_standalone_inference(
            checkpoint_path=CHECKPOINT,
            output_dir=OUTPUT_DIR,
            output_csv="submission.csv",
            batch_size=64,
            # dataset_name="dzungpham/SemEval-2026-TaskA-dataset",
            # dataset_config="",
            dataset_name="DaniilOr/SemEval-2026-Task13",
            dataset_config="A",
            split="test"
        )

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "dzungpham/graphcodebert-code-classification"
local_folder_path = "output_checkpoints"

api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

api.upload_folder(
    folder_path=local_folder_path,
    repo_id=repo_id,
    repo_type="model",
    commit_message="upload best checkpoints 200 with f1 score 0.68"
)

print(f"✅ Upload complete! View at: https://huggingface.co/{repo_id}")